# Rental Price Prediction Model
Training a Random Forest model to predict monthly rent for apartments in KL & Selangor.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [2]:
# Load data
df = pd.read_csv('../data/mudah-apartment-kl-selangor-cleaned.csv')
df.head()

,ads_id,prop_name,completion_year,monthly_rent,location,property_type,rooms,parking,bathroom,size (sq.ft),furnished,facilities,additional_facilities,region,facilities_count,additional_facilities_count
0,100323185,The Hipster @ Taman Desa,2022,4200,Kuala Lumpur - Taman Desa,Condominium,5,2,6,1842,Fully Furnished,"Minimart, Gymnasium, Security, Playground, Swi...","Air-Cond, Cooking Allowed, Washing Machine",Kuala Lumpur,10,3
1,100203973,Segar Courts,2017,2300,Kuala Lumpur - Cheras,Condominium,3,1,2,1170,Partially Furnished,"Playground, Parking, Barbeque area, Security, ...","Air-Cond, Cooking Allowed, Near KTM/LRT",Kuala Lumpur,9,3
2,100323128,Pangsapuri Teratak Muhibbah 2,2017,1000,Kuala Lumpur - Taman Desa,Apartment,3,0,2,650,Fully Furnished,"Minimart, Jogging Track, Lift, Swimming Pool",NaN,Kuala Lumpur,4,0
3,100191767,Sentul Point Suite Apartment,2020,1700,Kuala Lumpur - Sentul,Apartment,2,1,2,743,Partially Furnished,"Parking, Playground, Swimming Pool, Squash Cou...","Cooking Allowed, Near KTM/LRT, Washing Machine",Kuala Lumpur,8,3
4,97022692,Arte Mont Kiara,2017,1299,Kuala Lumpur - Mont Kiara,Service Residence,1,1,1,494,Not Furnished,"Parking, Security, Lift, Swimming Pool, Playgr...",Air-Cond,Kuala Lumpur,11,1


In [3]:
# Remove outliers - keep monthly_rent below 99th percentile
upper_limit = df["monthly_rent"].quantile(0.99)
print(f"Before removing outliers: {len(df)} rows")
print(f"99th percentile rent: RM {upper_limit:,.0f}")

df = df[df["monthly_rent"] <= upper_limit]
print(f"After removing outliers: {len(df)} rows")
print(f"Rent range: RM {df['monthly_rent'].min():,} - RM {df['monthly_rent'].max():,}")

Before removing outliers: 19043 rows
99th percentile rent: RM 4,800
After removing outliers: 18861 rows
Rent range: RM 70 - RM 4,800


In [4]:
# Select features and target, apply log transform to target
features = ["region", "location", "property_type", "rooms", "parking", "bathroom",
            "size (sq.ft)", "furnished", "facilities_count", "additional_facilities_count"]
target = "monthly_rent"

X = df[features].copy()
y = np.log1p(df[target])  # log transform to reduce skew

print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"Target (log-transformed) range: {y.min():.2f} - {y.max():.2f}")

Features: 10
Samples: 18861
Target (log-transformed) range: 4.26 - 8.48


In [5]:
# Encode categorical columns using LabelEncoder
label_encoders = {}
categorical_cols = ["region", "location", "property_type", "furnished"]

for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

region: {'Kuala Lumpur': np.int64(0), 'Selangor': np.int64(1)}
location: {'Kuala Lumpur - Ampang': np.int64(0), 'Kuala Lumpur - Ampang Hilir': np.int64(1), 'Kuala Lumpur - Bandar Damai Perdana': np.int64(2), 'Kuala Lumpur - Bandar Menjalara': np.int64(3), 'Kuala Lumpur - Bandar Tasik Selatan': np.int64(4), 'Kuala Lumpur - Bangsar': np.int64(5), 'Kuala Lumpur - Bangsar South': np.int64(6), 'Kuala Lumpur - Brickfields': np.int64(7), 'Kuala Lumpur - Bukit Bintang': np.int64(8), 'Kuala Lumpur - Bukit Jalil': np.int64(9), 'Kuala Lumpur - Bukit Tunku': np.int64(10), 'Kuala Lumpur - Chan Sow Lin': np.int64(11), 'Kuala Lumpur - Cheras': np.int64(12), 'Kuala Lumpur - City Centre': np.int64(13), 'Kuala Lumpur - Damansara': np.int64(14), 'Kuala Lumpur - Damansara Heights': np.int64(15), 'Kuala Lumpur - Desa Pandan': np.int64(16), 'Kuala Lumpur - Desa ParkCity': np.int64(17), 'Kuala Lumpur - Desa Petaling': np.int64(18), 'Kuala Lumpur - Gombak': np.int64(19), 'Kuala Lumpur - Jalan Ipoh': np.int64(

In [6]:
# Split data into train and test sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

Training set: 15088 samples
Test set: 3773 samples


In [7]:
# Train Random Forest model with tuned hyperparameters
model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [8]:
# Evaluate model (convert back from log scale for interpretable metrics)
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)  # reverse log transform
y_test_actual = np.expm1(y_test)  # reverse log transform

mae = mean_absolute_error(y_test_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))
r2 = r2_score(y_test_actual, y_pred)

print(f"MAE  (Mean Absolute Error): RM {mae:,.2f}")
print(f"RMSE (Root Mean Squared Error): RM {rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

MAE  (Mean Absolute Error): RM 208.93
RMSE (Root Mean Squared Error): RM 340.82
R² Score: 0.7540


In [9]:
# Feature importance
feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print(feature_importance.to_string(index=False))

                    Feature  Importance
               size (sq.ft)    0.317763
                  furnished    0.253163
                   location    0.165760
              property_type    0.098585
           facilities_count    0.059754
                      rooms    0.029726
                    parking    0.028997
additional_facilities_count    0.028001
                   bathroom    0.013587
                     region    0.004663


In [10]:
# Save model and label encoders
joblib.dump(model, '../model/random_forest_model.joblib')
joblib.dump(label_encoders, '../model/label_encoders.joblib')

print("Model and label encoders saved to ../model/")

Model and label encoders saved to ../model/
